# Koeltekaart Amsterdam — Unified CSS Analysis Pipeline

**Research questions**
- RQ1: Does social vulnerability stratify heat exposure across Amsterdam buurten, beyond physical predictors?
- RQ2: Are heat-vulnerable buurten systematically under-served by cooling infrastructure (double disadvantage)?

**Data**: 481 Amsterdam buurten · CBS Kerncijfers Wijken en Buurten · Klimaatrisicokaarten GDB · remotely sensed temp/NDVI

---

## 1. Setup

In [ ]:
# ── Standard imports ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings('ignore')

import os
import sys
from pathlib import Path
import json

import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.stats import shapiro, normaltest, jarque_bera, mstats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

# Optional spatial libraries – graceful fallback
try:
    import geopandas as gpd
    HAS_GEO = True
    print("geopandas available – spatial exports enabled")
except ImportError:
    HAS_GEO = False
    print("geopandas NOT available – running in CSV-only mode")

try:
    import libpysal
    from libpysal.weights import Queen
    from esda.moran import Moran, Moran_Local
    HAS_ESDA = True
    print("esda/libpysal available – spatial autocorrelation enabled")
except ImportError:
    HAS_ESDA = False
    print("esda/libpysal NOT available – install with: pip install esda libpysal")
    print("Spatial autocorrelation section will be skipped.")

try:
    import pingouin as pg
    HAS_PINGOUIN = True
    print("pingouin available")
except ImportError:
    HAS_PINGOUIN = False
    print("pingouin NOT available – KMO will use manual implementation")

# ── Global style ──────────────────────────────────────────────────────────────
AMS_BLUE   = '#004699'
AMS_RED    = '#ec0000'
AMS_GREEN  = '#00893c'
AMS_ORANGE = '#ff9100'
AMS_LIGHT  = '#009de6'

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'axes.edgecolor':   '#cccccc',
    'axes.spines.top':  False,
    'axes.spines.right': False,
    'font.family':      'sans-serif',
    'axes.titleweight': 'bold',
    'axes.titlesize':   13,
})

# ── Paths ─────────────────────────────────────────────────────────────────────
NB_DIR   = Path('.').resolve()          # notebook lives in Data_analysis/
DATA_DIR = NB_DIR / 'charlie'
OUT_DIR  = NB_DIR / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

GPKG_PATH = DATA_DIR / 'ams_features.gpkg'
CSV_PATH  = DATA_DIR / 'ams_features.csv'
GDB_PATH  = DATA_DIR / 'Klimaatrisicokaarten' / 'Risicokaarten definitief.gdb'
HVI_CSV   = OUT_DIR  / 'hvi_scores.csv'

print(f"\nData directory : {DATA_DIR}")
print(f"GPKG exists    : {GPKG_PATH.exists()}")
print(f"CSV exists     : {CSV_PATH.exists()}")
print(f"GDB exists     : {GDB_PATH.exists()}")
print(f"HVI CSV exists : {HVI_CSV.exists()}")

## 2. Data Loading & Cleaning

In [ ]:
# ── CBS missing-value sentinel codes ─────────────────────────────────────────
CBS_MISSING = [-100000000, -99999999, -99999997, -99999996]

def clean_cbs(df):
    """Replace CBS sentinel codes with NaN and coerce numeric columns."""
    for col in df.columns:
        if col in ('buurtcode', 'buurtnaam', 'wijkcode', 'gemeentecode',
                   'gemeentenaam', 'geometry'):
            continue
        try:
            s = pd.to_numeric(df[col], errors='coerce')
            s = s.replace(CBS_MISSING, np.nan)
            s[s < -9e7] = np.nan          # catch any further extreme negatives
            df[col] = s
        except Exception:
            pass
    return df

# ── Load primary dataset ──────────────────────────────────────────────────────
if HAS_GEO and GPKG_PATH.exists():
    print("Loading from GeoPackage …")
    gdf = gpd.read_file(GPKG_PATH)
    gdf = clean_cbs(gdf)
    df  = pd.DataFrame(gdf.drop(columns='geometry', errors='ignore'))
    print(f"  Loaded GeoPackage: {len(gdf)} buurten, {len(gdf.columns)} columns")
elif CSV_PATH.exists():
    print("Loading from CSV …")
    df = pd.read_csv(CSV_PATH, low_memory=False)
    df = clean_cbs(df)
    gdf = None
    print(f"  Loaded CSV: {len(df)} rows, {len(df.columns)} columns")
else:
    raise FileNotFoundError("Neither ams_features.gpkg nor ams_features.csv found in charlie/")

# ── Load Klimaatrisicokaarten GDB ─────────────────────────────────────────────
hi_df = None
if HAS_GEO and GDB_PATH.exists():
    try:
        layers = gpd.list_layers(str(GDB_PATH))
        print(f"\nGDB layers: {[l.name for l in layers.itertuples()]}")
        for row in layers.itertuples():
            lyr = gpd.read_file(str(GDB_PATH), layer=row.name)
            print(f"  Layer '{row.name}': {len(lyr)} rows, cols={list(lyr.columns)[:8]}")
            if any(c.startswith('HI_') for c in lyr.columns):
                hi_df = pd.DataFrame(lyr.drop(columns='geometry', errors='ignore'))
                print(f"  → Using layer '{row.name}' for heat-risk scores")
                break
    except Exception as e:
        print(f"  Could not read GDB: {e}")

# Fallback: use hvi_scores.csv for HI_ columns
if hi_df is None and HVI_CSV.exists():
    print("\nLoading HI scores from outputs/hvi_scores.csv …")
    hi_df = pd.read_csv(HVI_CSV)
    print(f"  {len(hi_df)} rows, cols={list(hi_df.columns)}")

# ── Join HI scores onto main dataframe ───────────────────────────────────────
HI_COLS = ['HI_TOTAAL_S', 'HI_BLOOTSTELLING_S', 'HI_GEVOELIGHEID_S']
if hi_df is not None:
    join_key_df  = 'buurtnaam'
    join_key_hi  = 'buurtnaam' if 'buurtnaam' in hi_df.columns else hi_df.columns[0]
    cols_to_add  = [join_key_hi] + [c for c in HI_COLS if c in hi_df.columns]
    df = df.merge(hi_df[cols_to_add].drop_duplicates(subset=join_key_hi),
                  left_on=join_key_df, right_on=join_key_hi, how='left')
    print(f"\nAfter HI join: {len(df)} rows, {len(df.columns)} columns")
    print(f"HI_TOTAAL_S non-null: {df['HI_TOTAAL_S'].notna().sum() if 'HI_TOTAAL_S' in df.columns else 0}")

# ── Key variable groups ───────────────────────────────────────────────────────
PHYS_VARS = [
    'temp_mean', 'ndvi_mean', 'water_prc', 'road_prc',
    'bevolkingsdichtheidInwonersPerKm2'
]
SOC_VARS = [
    'percentagePersonen65JaarEnOuder',
    'percentageEenpersoonshuishoudens',
    'aantalWmoClientenPer1000Inwoners',
    'percentagePersonenMetLaagInkomen',
    'percentageHuishoudensOnderOfRondSociaalMinimum',
    'percentageNietWesterseMigratieachtergrond'
]
COOL_VARS = [
    'bibliotheekGemiddeldeAfstandInKm',
    'zwembadGemiddeldeAfstandInKm',
    'treinstationGemiddeldeAfstandInKm',
    'huisartsenpraktijkGemiddeldeAfstandInKm'
]
CTRL_VARS = [
    'percentageBouwjaarklasseVanaf2000',
    'gemiddeldElektriciteitsverbruikTotaal'
]
ALL_KEY = PHYS_VARS + SOC_VARS + COOL_VARS + CTRL_VARS

# Ensure all key vars exist (some may be absent in certain data versions)
ALL_KEY = [v for v in ALL_KEY if v in df.columns]
print(f"\nKey variables available: {len(ALL_KEY)}/{len(PHYS_VARS+SOC_VARS+COOL_VARS+CTRL_VARS)}")

# ── Missing-value report ──────────────────────────────────────────────────────
miss = (df[ALL_KEY].isna().sum()
        .rename('n_missing')
        .reset_index()
        .rename(columns={'index': 'variable'}))
miss['pct_missing'] = (miss['n_missing'] / len(df) * 100).round(1)
miss = miss.sort_values('pct_missing', ascending=False)
print("\nMissing values per key variable:")
print(miss.to_string(index=False))

## 3. Descriptive Statistics & EDA

In [ ]:
# ── Summary statistics ───────────────────────────────────────────────────────
desc = df[ALL_KEY].describe().T
desc['cv'] = (desc['std'] / desc['mean'].abs()).round(3)
print("Descriptive statistics (all key variables):")
with pd.option_context('display.float_format', '{:.3f}'.format, 'display.max_rows', 50):
    display(desc)

In [ ]:
# ── Per-variable histograms with KDE ─────────────────────────────────────────
n_vars = len(ALL_KEY)
ncols  = 3
nrows  = int(np.ceil(n_vars / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 3.5*nrows))
axes = axes.flatten()

for i, var in enumerate(ALL_KEY):
    ax = axes[i]
    valid = df[var].dropna()
    ax.hist(valid, bins=30, color=AMS_BLUE, alpha=0.65, edgecolor='white',
            density=True, label='Histogram')
    # KDE
    if len(valid) > 5:
        from scipy.stats import gaussian_kde
        xs = np.linspace(valid.min(), valid.max(), 200)
        kde = gaussian_kde(valid, bw_method='scott')
        ax.plot(xs, kde(xs), color=AMS_RED, lw=2, label='KDE')
    short = var.replace('Gemiddelde', '').replace('percentage', 'pct_')[:30]
    ax.set_title(short, fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7)

# Hide unused axes
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Distribution of Key Variables (n=481 buurten)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_histograms.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/eda_histograms.png")

In [ ]:
# ── Box plots (standardised) ──────────────────────────────────────────────────
scaler = StandardScaler()
df_std = pd.DataFrame(
    scaler.fit_transform(df[ALL_KEY].fillna(df[ALL_KEY].median())),
    columns=ALL_KEY
)

fig, ax = plt.subplots(figsize=(14, 6))
df_std[ALL_KEY].boxplot(ax=ax, notch=False,
    boxprops=dict(color=AMS_BLUE),
    whiskerprops=dict(color=AMS_BLUE),
    capprops=dict(color=AMS_BLUE),
    medianprops=dict(color=AMS_RED, lw=2),
    flierprops=dict(marker='o', markersize=3, alpha=0.3, color=AMS_ORANGE))
ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.set_xticklabels(
    [v.replace('Gemiddelde', '').replace('percentage', 'pct_')[:18]
     for v in ALL_KEY],
    rotation=60, ha='right', fontsize=8
)
ax.set_ylabel('z-score')
ax.set_title('Standardised Box Plots — Key Variables')
plt.tight_layout()
plt.savefig(OUT_DIR / 'eda_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/eda_boxplots.png")

In [ ]:
# ── Value range table ─────────────────────────────────────────────────────────
range_tbl = pd.DataFrame({
    'Variable'  : ALL_KEY,
    'Min'       : df[ALL_KEY].min().round(3).values,
    'P25'       : df[ALL_KEY].quantile(.25).round(3).values,
    'Median'    : df[ALL_KEY].median().round(3).values,
    'P75'       : df[ALL_KEY].quantile(.75).round(3).values,
    'Max'       : df[ALL_KEY].max().round(3).values,
    'IQR'       : (df[ALL_KEY].quantile(.75) - df[ALL_KEY].quantile(.25)).round(3).values,
    'Skew'      : df[ALL_KEY].skew().round(3).values,
    'Kurt'      : df[ALL_KEY].kurt().round(3).values,
})
with pd.option_context('display.max_rows', 50, 'display.float_format', '{:.3f}'.format):
    display(range_tbl)

## 4. Normality Testing

In [ ]:
# ── Shapiro-Wilk, D'Agostino K², Jarque-Bera ──────────────────────────────────
results = []
alpha = 0.05

for var in ALL_KEY:
    s = df[var].dropna()
    n = len(s)

    # Shapiro-Wilk (reliable up to n=5000)
    if n <= 5000:
        sw_stat, sw_p = shapiro(s)
    else:
        sw_stat, sw_p = np.nan, np.nan

    # D'Agostino K²
    dk_stat, dk_p = normaltest(s)

    # Jarque-Bera
    jb_stat, jb_p = jarque_bera(s)

    results.append({
        'Variable'   : var,
        'n'          : n,
        'SW_W'       : round(sw_stat, 4) if not np.isnan(sw_stat) else 'n/a',
        'SW_p'       : round(sw_p, 4)   if not np.isnan(sw_p)    else 'n/a',
        'DK_K2'      : round(dk_stat, 3),
        'DK_p'       : round(dk_p,   4),
        'JB_stat'    : round(jb_stat, 3),
        'JB_p'       : round(jb_p,   4),
        'Normal_α05' : ('Yes' if (sw_p > alpha if isinstance(sw_p, float) else True)
                               and dk_p > alpha and jb_p > alpha else 'No')
    })

norm_df = pd.DataFrame(results)
print("Normality test results (α = 0.05):")
with pd.option_context('display.max_rows', 50):
    display(norm_df)

n_normal = (norm_df['Normal_α05'] == 'Yes').sum()
print(f"\n{n_normal}/{len(norm_df)} variables pass all three normality tests at α=0.05")
print("Interpretation: Most socioeconomic variables are right-skewed; non-parametric "
      "or robust methods are preferred for inference.")

In [ ]:
# ── QQ plots for key variables ────────────────────────────────────────────────
qq_vars = PHYS_VARS[:5] + SOC_VARS[:4]   # pick 9 representative vars
ncols = 3
nrows = int(np.ceil(len(qq_vars) / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(5*ncols, 4*nrows))
axes = axes.flatten()

for i, var in enumerate(qq_vars):
    ax = axes[i]
    s = df[var].dropna()
    (osm, osr), (slope, intercept, r) = stats.probplot(s, dist='norm')
    ax.scatter(osm, osr, s=8, alpha=0.6, color=AMS_BLUE)
    x_line = np.array([osm.min(), osm.max()])
    ax.plot(x_line, slope*x_line + intercept, color=AMS_RED, lw=2)
    short = var.replace('percentage', 'pct_').replace('Gemiddelde', '')[:25]
    ax.set_title(f'{short}\nr={r:.3f}', fontsize=9)
    ax.set_xlabel('Theoretical quantiles', fontsize=8)
    ax.set_ylabel('Sample quantiles', fontsize=8)
    ax.tick_params(labelsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('QQ Plots — Key Variables vs. Normal Distribution', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'normality_qq.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/normality_qq.png")

## 5. Outlier Detection

In [ ]:
# ── IQR method ───────────────────────────────────────────────────────────────
outlier_flags = pd.DataFrame(index=df.index)

for var in ALL_KEY:
    s = df[var]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5*iqr, q3 + 1.5*iqr
    outlier_flags[f'iqr_{var}'] = (s < lo) | (s > hi)

# ── Z-score method ────────────────────────────────────────────────────────────
for var in ALL_KEY:
    s = df[var].fillna(df[var].median())
    z = np.abs(stats.zscore(s))
    outlier_flags[f'z_{var}'] = z > 3

# Count how many variables each buurt is an outlier on
iqr_cols = [c for c in outlier_flags.columns if c.startswith('iqr_')]
z_cols   = [c for c in outlier_flags.columns if c.startswith('z_')]

df['n_iqr_outliers'] = outlier_flags[iqr_cols].sum(axis=1)
df['n_z_outliers']   = outlier_flags[z_cols  ].sum(axis=1)

THRESH = 3
iqr_multi = df[df['n_iqr_outliers'] >= THRESH][['buurtnaam','n_iqr_outliers']].sort_values(
    'n_iqr_outliers', ascending=False)
z_multi   = df[df['n_z_outliers']   >= THRESH][['buurtnaam','n_z_outliers'  ]].sort_values(
    'n_z_outliers',   ascending=False)

print(f"Buurten that are IQR outliers on ≥{THRESH} variables ({len(iqr_multi)}):")
print(iqr_multi.to_string(index=False))
print(f"\nBuurten that are Z-score outliers on ≥{THRESH} variables ({len(z_multi)}):")
print(z_multi.to_string(index=False))

# ── Outlier heatmap ───────────────────────────────────────────────────────────
top_outlier_buurten = iqr_multi.head(20)['buurtnaam']
subset = df[df['buurtnaam'].isin(top_outlier_buurten)].set_index('buurtnaam')

# Z-scores for display
z_mat = pd.DataFrame(
    stats.zscore(df[ALL_KEY].fillna(df[ALL_KEY].median()), nan_policy='omit'),
    index=df.index, columns=ALL_KEY
).loc[subset.index]

short_cols = [c.replace('percentage','pct_').replace('Gemiddelde','')[:18] for c in ALL_KEY]
z_mat.columns = short_cols

fig, ax = plt.subplots(figsize=(16, 7))
sns.heatmap(z_mat, cmap='RdBu_r', center=0, linewidths=0.3,
            vmin=-4, vmax=4, ax=ax, cbar_kws={'label': 'z-score'})
ax.set_title('Z-score Heatmap — Top 20 Outlier Buurten')
ax.set_ylabel('')
ax.tick_params(axis='x', rotation=60, labelsize=7)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'outlier_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/outlier_heatmap.png")

## 6. Correlation & Covariance Analysis

In [ ]:
# ── Significance stars helper ─────────────────────────────────────────────────
def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    return ''

# ── Pearson & Spearman correlation matrices ───────────────────────────────────
sub = df[ALL_KEY].dropna()
n_sub = len(sub)
print(f"Correlation computed on {n_sub} complete-case observations")

pearson_r  = sub.corr(method='pearson')
spearman_r = sub.corr(method='spearman')

# p-values for Pearson
from scipy.stats import pearsonr, spearmanr as sp_rank
n_v = len(ALL_KEY)
p_pearson  = np.ones((n_v, n_v))
p_spearman = np.ones((n_v, n_v))
for i in range(n_v):
    for j in range(n_v):
        if i != j:
            _, p_pearson[i,j]  = pearsonr(sub.iloc[:,i], sub.iloc[:,j])
            _, p_spearman[i,j] = sp_rank(sub.iloc[:,i], sub.iloc[:,j])

# ── Side-by-side heatmaps ─────────────────────────────────────────────────────
short_labels = [c.replace('percentage','pct_').replace('Gemiddelde','')[:18]
                for c in ALL_KEY]

def annotate_sig(ax, mat, p_mat, threshold=0.05):
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            stars = sig_stars(p_mat[i,j])
            if stars:
                ax.text(j+0.5, i+0.5, stars, ha='center', va='center',
                        fontsize=6, color='black')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))

mask_upper = np.triu(np.ones_like(pearson_r, dtype=bool))

sns.heatmap(pearson_r, mask=mask_upper, annot=False, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=short_labels, yticklabels=short_labels,
            linewidths=0.3, ax=ax1, cbar_kws={'shrink':0.7})
ax1.set_title("Pearson r  (* p<.05  ** p<.01  *** p<.001)")
ax1.tick_params(axis='x', rotation=60, labelsize=7)
ax1.tick_params(axis='y', labelsize=7)
annotate_sig(ax1, pearson_r.values, p_pearson)

sns.heatmap(spearman_r, mask=mask_upper, annot=False, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            xticklabels=short_labels, yticklabels=short_labels,
            linewidths=0.3, ax=ax2, cbar_kws={'shrink':0.7})
ax2.set_title("Spearman ρ  (* p<.05  ** p<.01  *** p<.001)")
ax2.tick_params(axis='x', rotation=60, labelsize=7)
ax2.tick_params(axis='y', labelsize=7)
annotate_sig(ax2, spearman_r.values, p_spearman)

plt.suptitle('Correlation Matrices — Key Variables', fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / 'correlation_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/correlation_heatmaps.png")

In [ ]:
# ── Kendall's tau for SVI variables ──────────────────────────────────────────
from scipy.stats import kendalltau

print("Kendall's tau — pairwise among social vulnerability variables:")
print(f"{'Variable 1':<45} {'Variable 2':<45} {'tau':>7} {'p':>10}")
print('-'*110)
for i, v1 in enumerate(SOC_VARS):
    if v1 not in sub.columns: continue
    for v2 in SOC_VARS[i+1:]:
        if v2 not in sub.columns: continue
        tau, p = kendalltau(sub[v1], sub[v2])
        stars = sig_stars(p)
        print(f"{v1[:44]:<45} {v2[:44]:<45} {tau:>7.3f} {p:>10.4f} {stars}")

# ── Covariance matrix (standardised) ─────────────────────────────────────────
cov_std = sub[SOC_VARS].apply(stats.zscore).cov()
fig, ax = plt.subplots(figsize=(8, 6))
short_soc = [c.replace('percentage','pct_').replace('Gemiddelde','')[:22] for c in SOC_VARS if c in sub.columns]
soc_avail = [c for c in SOC_VARS if c in sub.columns]
sns.heatmap(sub[soc_avail].apply(stats.zscore).cov(),
            annot=True, fmt='.2f', cmap='Blues',
            xticklabels=short_soc, yticklabels=short_soc,
            linewidths=0.4, ax=ax)
ax.set_title('Covariance Matrix — SVI Variables (standardised)')
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.tick_params(axis='y', labelsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / 'covariance_svi.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/covariance_svi.png")

## 7. Multicollinearity (VIF)

In [ ]:
# ── Variance Inflation Factor ─────────────────────────────────────────────────
predictor_vars = PHYS_VARS + SOC_VARS + CTRL_VARS
predictor_vars = [v for v in predictor_vars if v in df.columns]

# Use median imputation for VIF computation
X_vif = df[predictor_vars].copy()
for col in X_vif.columns:
    X_vif[col] = X_vif[col].fillna(X_vif[col].median())

X_vif = sm.add_constant(X_vif.astype(float))

vif_data = pd.DataFrame()
vif_data['Variable'] = X_vif.columns
vif_data['VIF'] = [variance_inflation_factor(X_vif.values, i)
                   for i in range(X_vif.shape[1])]
vif_data = vif_data[vif_data['Variable'] != 'const']
vif_data = vif_data.sort_values('VIF', ascending=False)
vif_data['Flag'] = vif_data['VIF'].apply(lambda x: '⚠ DROP?' if x > 10 else ('Caution' if x > 5 else 'OK'))

print("VIF results (VIF > 10 → multicollinearity concern):")
with pd.option_context('display.float_format', '{:.2f}'.format):
    display(vif_data)

# ── VIF bar chart ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))
colors = [AMS_RED if v > 10 else (AMS_ORANGE if v > 5 else AMS_BLUE) for v in vif_data['VIF']]
short_names = [c.replace('percentage','pct_').replace('Gemiddelde','')[:22] for c in vif_data['Variable']]
ax.barh(short_names, vif_data['VIF'], color=colors)
ax.axvline(10, color=AMS_RED,    lw=2, ls='--', label='VIF = 10 (high)')
ax.axvline(5,  color=AMS_ORANGE, lw=2, ls=':',  label='VIF = 5 (moderate)')
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factor — Predictor Variables')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'vif_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/vif_chart.png")

high_vif = vif_data[vif_data['VIF'] > 10]['Variable'].tolist()
print(f"\nSuggested drops (VIF > 10): {high_vif}")
print("Interpretation: Variables with VIF > 10 should be dropped or combined; "
      "prefer keeping theoretically central predictors.")

## 8. Social Vulnerability Index (PCA)

In [ ]:
# ── Prepare SVI matrix ────────────────────────────────────────────────────────
soc_avail = [v for v in SOC_VARS if v in df.columns]
X_soc_raw = df[soc_avail].copy()

# Median imputation for PCA
for col in X_soc_raw.columns:
    X_soc_raw[col] = X_soc_raw[col].fillna(X_soc_raw[col].median())

scaler_soc = StandardScaler()
X_soc = scaler_soc.fit_transform(X_soc_raw)
X_soc_df = pd.DataFrame(X_soc, columns=soc_avail, index=df.index)

# ── KMO (Kaiser-Meyer-Olkin) ─────────────────────────────────────────────────
def kmo(X):
    """Manual KMO implementation."""
    corr = np.corrcoef(X.T)
    try:
        inv_corr = np.linalg.inv(corr)
    except np.linalg.LinAlgError:
        return np.nan, np.nan * np.ones(X.shape[1])
    # Partial correlations
    p = -inv_corr / np.sqrt(np.outer(np.diag(inv_corr), np.diag(inv_corr)))
    np.fill_diagonal(p, 0)
    np.fill_diagonal(corr, 0)
    r2 = corr**2
    p2 = p**2
    kmo_num = r2.sum()
    kmo_den = r2.sum() + p2.sum()
    kmo_overall = kmo_num / kmo_den if kmo_den > 0 else np.nan
    kmo_per_item = r2.sum(axis=0) / (r2.sum(axis=0) + p2.sum(axis=0))
    return kmo_overall, kmo_per_item

if HAS_PINGOUIN:
    kmo_result = pg.kmo(X_soc_df)
    kmo_val = kmo_result.kmo
    print(f"KMO (pingouin): {kmo_val:.4f}")
else:
    kmo_val, kmo_items = kmo(X_soc)
    print(f"KMO (manual): {kmo_val:.4f}")
    for name, v in zip(soc_avail, kmo_items):
        short = name.replace('percentage','pct_')[:35]
        print(f"  {short:<36}: {v:.4f}")

kmo_interp = ('Marvellous' if kmo_val >= 0.9 else
               'Meritorious' if kmo_val >= 0.8 else
               'Middling' if kmo_val >= 0.7 else
               'Mediocre' if kmo_val >= 0.6 else 'Unacceptable')
print(f"KMO interpretation: {kmo_interp}")

# ── Bartlett's sphericity test ─────────────────────────────────────────────────
n_obs, n_feat = X_soc.shape
corr_mat = np.corrcoef(X_soc.T)
det = np.linalg.det(corr_mat)
if det <= 0:
    det = 1e-300
chi2_b = -((n_obs - 1) - (2*n_feat + 5)/6) * np.log(det)
df_b = n_feat*(n_feat-1)//2
p_bart = 1 - stats.chi2.cdf(chi2_b, df_b)
print(f"\nBartlett's sphericity: χ²({df_b}) = {chi2_b:.2f}, p = {p_bart:.6f}")
if p_bart < 0.05:
    print("→ Significant (p<0.05): correlation matrix is not identity → PCA is appropriate.")
else:
    print("→ Non-significant: PCA may not be appropriate.")

In [ ]:
# ── Run PCA ───────────────────────────────────────────────────────────────────
pca = PCA()
pca.fit(X_soc)
explained_var  = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)
loadings       = pd.DataFrame(
    pca.components_.T,
    index=soc_avail,
    columns=[f'PC{i+1}' for i in range(len(soc_avail))]
)

print("PCA Explained Variance:")
for i, (ev, cv) in enumerate(zip(explained_var, cumulative_var)):
    bar = '█' * int(ev*60)
    print(f"  PC{i+1}: {ev*100:5.1f}%  cumulative={cv*100:5.1f}%  {bar}")

n_components_80 = int(np.argmax(cumulative_var >= 0.80)) + 1
print(f"\nComponents needed for ≥80% variance: {n_components_80}")

# ── Scree plot + cumulative variance ─────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.bar(range(1, len(explained_var)+1), explained_var*100, color=AMS_BLUE, alpha=0.8)
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained variance (%)')
ax1.set_title('Scree Plot')
ax1.axhline(y=100/len(soc_avail), color=AMS_RED, ls='--',
            label=f'Mean ({100/len(soc_avail):.1f}%)')
ax1.legend()

ax2.plot(range(1, len(cumulative_var)+1), cumulative_var*100,
         'o-', color=AMS_BLUE, lw=2)
ax2.axhline(80, color=AMS_ORANGE, ls='--', label='80%')
ax2.axhline(90, color=AMS_RED,    ls='--', label='90%')
ax2.fill_between(range(1, len(cumulative_var)+1), cumulative_var*100,
                 alpha=0.15, color=AMS_BLUE)
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Explained Variance (%)')
ax2.set_title('Cumulative Variance Explained')
ax2.legend()
ax2.set_ylim(0, 105)

plt.suptitle('PCA — Social Vulnerability Variables', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/pca_scree.png")

In [ ]:
# ── Loading heatmap ───────────────────────────────────────────────────────────
short_soc = [c.replace('percentage','pct_').replace('aantaWmo','wmo_')[:25]
             for c in soc_avail]
loadings_plot = loadings.copy()
loadings_plot.index = short_soc

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(loadings_plot.iloc[:, :4], annot=True, fmt='.3f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.4, ax=ax, cbar_kws={'label': 'Loading'})
ax.set_title('PCA Loadings — First 4 Components (SVI)')
ax.set_xlabel('Component')
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/pca_loadings.png")

# ── Biplot (PC1 vs PC2) ───────────────────────────────────────────────────────
scores = pca.transform(X_soc)
fig, ax = plt.subplots(figsize=(9, 8))
ax.scatter(scores[:,0], scores[:,1], s=18, alpha=0.5, color=AMS_BLUE, label='Buurten')

scale = 3.5
for i, var in enumerate(soc_avail):
    ax.annotate('', xy=(pca.components_[0,i]*scale, pca.components_[1,i]*scale),
                xytext=(0,0),
                arrowprops=dict(arrowstyle='->', color=AMS_RED, lw=1.8))
    ax.text(pca.components_[0,i]*scale*1.1, pca.components_[1,i]*scale*1.1,
            var.replace('percentage','pct_')[:20], fontsize=8, color=AMS_RED)

ax.axhline(0, color='grey', lw=0.8, ls='--')
ax.axvline(0, color='grey', lw=0.8, ls='--')
ax.set_xlabel(f'PC1 ({explained_var[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({explained_var[1]*100:.1f}%)')
ax.set_title('PCA Biplot — Social Vulnerability Variables')
plt.tight_layout()
plt.savefig(OUT_DIR / 'pca_biplot.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/pca_biplot.png")

# ── Assign SVI score (PC1, flipped so higher = more vulnerable) ───────────────
pc1_raw = scores[:, 0]
# Check sign: if positive loadings for vulnerability vars → keep; else flip
# High percentagePersonen65JaarEnOuder should → high SVI
if 'percentagePersonen65JaarEnOuder' in soc_avail:
    idx_65 = soc_avail.index('percentagePersonen65JaarEnOuder')
    if loadings.loc['percentagePersonen65JaarEnOuder', 'PC1'] < 0:
        pc1_raw = -pc1_raw

# Normalise to 0–1
svi_min, svi_max = pc1_raw.min(), pc1_raw.max()
df['svi_pca'] = (pc1_raw - svi_min) / (svi_max - svi_min)

print(f"\nSVI (PC1) computed: min={df['svi_pca'].min():.3f}, "
      f"max={df['svi_pca'].max():.3f}, mean={df['svi_pca'].mean():.3f}")
print(f"Highest SVI buurten:")
print(df.nlargest(5, 'svi_pca')[['buurtnaam','svi_pca']].to_string(index=False))

## 9. RQ1 — Heat Stratification

In [ ]:
# ── Prepare regression dataset ────────────────────────────────────────────────
reg_vars = ['temp_mean'] + PHYS_VARS[1:] + SOC_VARS + CTRL_VARS + ['svi_pca']
reg_vars = [v for v in reg_vars if v in df.columns]
df_reg = df[reg_vars + ['buurtnaam']].copy()

for col in reg_vars:
    df_reg[col] = pd.to_numeric(df_reg[col], errors='coerce')
    df_reg[col] = df_reg[col].fillna(df_reg[col].median())

y = df_reg['temp_mean']

# ── Model A: Physical predictors only ────────────────────────────────────────
phys_avail = [v for v in PHYS_VARS[1:] + CTRL_VARS if v in df_reg.columns]
X_phys = sm.add_constant(df_reg[phys_avail].astype(float))
ols_a  = sm.OLS(y, X_phys).fit(cov_type='HC3')

# ── Model B: Physical + Social ────────────────────────────────────────────────
soc_reg  = [v for v in SOC_VARS if v in df_reg.columns]
X_full   = sm.add_constant(df_reg[phys_avail + soc_reg].astype(float))
ols_b    = sm.OLS(y, X_full).fit(cov_type='HC3')

print("=== Model A: Physical Predictors Only (HC3 robust SEs) ===")
print(ols_a.summary2())
print(f"\n=== Model B: Physical + Social Predictors (HC3 robust SEs) ===")
print(ols_b.summary2())

print(f"\n--- Model Comparison ---")
print(f"Model A  R² = {ols_a.rsquared:.4f},  adj-R² = {ols_a.rsquared_adj:.4f},  AIC = {ols_a.aic:.1f}")
print(f"Model B  R² = {ols_b.rsquared:.4f},  adj-R² = {ols_b.rsquared_adj:.4f},  AIC = {ols_b.aic:.1f}")
print(f"ΔR² (social contribution) = {ols_b.rsquared - ols_a.rsquared:.4f}")

if ols_b.rsquared - ols_a.rsquared > 0.02:
    print("Interpretation: Social predictors explain meaningful additional variance in surface temperature.")
else:
    print("Interpretation: Social predictors add modest incremental variance beyond physical factors.")

In [ ]:
# ── Standardised β coefficients ──────────────────────────────────────────────
def standardised_betas(ols_result, y_series, X_df):
    """Return standardised regression coefficients."""
    y_std = y_series.std()
    betas = {}
    for col in X_df.columns:
        if col == 'const':
            continue
        b = ols_result.params[col]
        x_std = X_df[col].std()
        betas[col] = b * (x_std / y_std)
    return pd.Series(betas).sort_values()

beta_a = standardised_betas(ols_a, y, X_phys)
beta_b = standardised_betas(ols_b, y, X_full)

print("Standardised β — Model A (Physical Only):")
print(beta_a.round(4).to_string())
print("\nStandardised β — Model B (Full):")
print(beta_b.round(4).to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
short_a = [c.replace('percentage','pct_').replace('Gemiddelde','')[:20] for c in beta_a.index]
short_b = [c.replace('percentage','pct_').replace('Gemiddelde','')[:20] for c in beta_b.index]

colors_a = [AMS_RED if v < 0 else AMS_BLUE for v in beta_a.values]
colors_b = [AMS_RED if v < 0 else AMS_BLUE for v in beta_b.values]

ax1.barh(short_a, beta_a.values, color=colors_a)
ax1.axvline(0, color='black', lw=0.8)
ax1.set_title('Model A — Physical Only
Standardised β')
ax1.set_xlabel('β')

ax2.barh(short_b, beta_b.values, color=colors_b)
ax2.axvline(0, color='black', lw=0.8)
ax2.set_title('Model B — Physical + Social
Standardised β')
ax2.set_xlabel('β')

plt.suptitle('RQ1: Standardised Regression Coefficients (HC3)', fontsize=13)
plt.tight_layout()
plt.savefig(OUT_DIR / 'rq1_betas.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/rq1_betas.png")

# ── Residuals plot ────────────────────────────────────────────────────────────
residuals_b = ols_b.resid
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(ols_b.fittedvalues, residuals_b, s=10, alpha=0.5, color=AMS_BLUE)
axes[0].axhline(0, color=AMS_RED, lw=1.5, ls='--')
axes[0].set_xlabel('Fitted values'); axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted')

stats.probplot(residuals_b, plot=axes[1])
axes[1].set_title('QQ Plot of Residuals')

axes[2].hist(residuals_b, bins=30, color=AMS_BLUE, alpha=0.7, edgecolor='white')
axes[2].set_xlabel('Residual'); axes[2].set_ylabel('Count')
axes[2].set_title('Residual Distribution')

dw = durbin_watson(residuals_b)
print(f"\nDurbin-Watson statistic: {dw:.4f}  (2.0 = no autocorrelation)")

plt.tight_layout()
plt.savefig(OUT_DIR / 'rq1_residuals.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/rq1_residuals.png")

In [ ]:
# ── Moran's I on residuals (if geo available) ─────────────────────────────────
if HAS_GEO and HAS_ESDA and gdf is not None:
    try:
        gdf_reg = gdf.copy()
        gdf_reg['ols_resid'] = np.nan
        gdf_reg.loc[df_reg.index, 'ols_resid'] = residuals_b.values
        gdf_reg = gdf_reg.dropna(subset=['ols_resid'])

        w_q = Queen.from_dataframe(gdf_reg, silence_warnings=True)
        w_q.transform = 'r'

        mi_resid = Moran(gdf_reg['ols_resid'].values, w_q, permutations=999)
        print(f"Moran's I on OLS residuals: I={mi_resid.I:.4f}, "
              f"E[I]={mi_resid.EI:.4f}, p={mi_resid.p_sim:.4f}, z={mi_resid.z_sim:.4f}")
        if mi_resid.p_sim < 0.05:
            print("→ Significant spatial autocorrelation in residuals — "
                  "spatial lag or error model warranted.")
        else:
            print("→ Residuals show no significant spatial autocorrelation.")
    except Exception as e:
        print(f"Moran's I on residuals failed: {e}")
else:
    print("Skipping residual Moran's I (requires geopandas + esda + geometry).")

## 10. Spatial Autocorrelation (Global + LISA)

In [ ]:
if not HAS_ESDA:
    print("esda/libpysal not installed. Install with:")
    print("  pip install esda libpysal")
    print("Skipping spatial autocorrelation section.")
elif not HAS_GEO or gdf is None:
    print("geopandas or geometry not available. Skipping spatial section.")
else:
    # ── Build spatial weights ─────────────────────────────────────────────────
    gdf_sp = gdf.copy()

    # Attach SVI and temp_mean
    gdf_sp['svi_pca']  = df['svi_pca'].values
    gdf_sp['temp_mean'] = df['temp_mean'].values

    gdf_sp = gdf_sp.dropna(subset=['svi_pca', 'temp_mean'])
    w_queen = Queen.from_dataframe(gdf_sp, silence_warnings=True)
    w_queen.transform = 'r'
    print(f"Queen contiguity weights: {w_queen.n} units, "
          f"avg {w_queen.mean_neighbors:.1f} neighbours")

    # ── Global Moran's I ──────────────────────────────────────────────────────
    for varname in ['temp_mean', 'svi_pca']:
        mi = Moran(gdf_sp[varname].values, w_queen, permutations=999)
        print(f"\nGlobal Moran's I [{varname}]:")
        print(f"  I = {mi.I:.4f},  E[I] = {mi.EI:.4f},  "
              f"z = {mi.z_sim:.4f},  p (pseudo) = {mi.p_sim:.4f}")
        if mi.p_sim < 0.05:
            direction = 'positive' if mi.I > 0 else 'negative'
            print(f"  → Significant {direction} spatial autocorrelation (p<0.05, 999 permutations).")
        else:
            print("  → No significant spatial autocorrelation.")

    # ── LISA (Local Moran) ────────────────────────────────────────────────────
    lisa = Moran_Local(gdf_sp['temp_mean'].values, w_queen, permutations=999)
    gdf_sp['lisa_I']    = lisa.Is
    gdf_sp['lisa_p']    = lisa.p_sim
    gdf_sp['lisa_q']    = lisa.q

    # Classify: 1=HH, 2=LH, 3=LL, 4=HL (esda convention)
    sig_mask = gdf_sp['lisa_p'] < 0.05
    cluster_labels = {1:'HH (hot spot)', 2:'LH', 3:'LL (cold spot)', 4:'HL', 0:'n.s.'}
    gdf_sp['lisa_cluster'] = 'n.s.'
    gdf_sp.loc[sig_mask, 'lisa_cluster'] = gdf_sp.loc[sig_mask, 'lisa_q'].map(cluster_labels)

    print("\nLISA cluster distribution (temp_mean, p<0.05):")
    print(gdf_sp['lisa_cluster'].value_counts().to_string())

    # ── LISA map ──────────────────────────────────────────────────────────────
    cluster_colors = {
        'HH (hot spot)' : '#d7191c',
        'HL'            : '#fdae61',
        'LH'            : '#abd9e9',
        'LL (cold spot)': '#2c7bb6',
        'n.s.'          : '#f0f0f0'
    }
    gdf_sp['color'] = gdf_sp['lisa_cluster'].map(cluster_colors).fillna('#f0f0f0')

    fig, ax = plt.subplots(figsize=(10, 10))
    gdf_sp.plot(color=gdf_sp['color'], linewidth=0.3, edgecolor='white', ax=ax)
    patches = [mpatches.Patch(color=v, label=k) for k, v in cluster_colors.items()]
    ax.legend(handles=patches, loc='upper left', fontsize=9, title='LISA Cluster')
    ax.set_title('LISA Cluster Map — Surface Temperature (temp_mean)', fontsize=13)
    ax.set_axis_off()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'lisa_map.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/lisa_map.png")

    # Store cluster on df for export
    df['lisa_cluster'] = gdf_sp['lisa_cluster'].values if len(gdf_sp) == len(df) else np.nan

## 11. RQ2 — Cooling Access & Double Disadvantage

In [ ]:
# ── Composite cooling access score ───────────────────────────────────────────
cool_avail = [v for v in COOL_VARS if v in df.columns]
print(f"Cooling variables used: {cool_avail}")

df_cool = df[cool_avail].copy()
for col in df_cool.columns:
    df_cool[col] = pd.to_numeric(df_cool[col], errors='coerce').fillna(df_cool[col].median())

# Invert so higher = better access, then normalise to 0-1
for col in df_cool.columns:
    mx = df_cool[col].max()
    mn = df_cool[col].min()
    if mx > mn:
        df_cool[col] = 1 - (df_cool[col] - mn) / (mx - mn)
    else:
        df_cool[col] = 0.5

df['cooling_access'] = df_cool.mean(axis=1)

print(f"\nCooling access score: min={df['cooling_access'].min():.3f}, "
      f"max={df['cooling_access'].max():.3f}, mean={df['cooling_access'].mean():.3f}")

# ── Scatter: SVI vs cooling access ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 7))
sc = ax.scatter(df['svi_pca'], df['cooling_access'],
                s=28, alpha=0.6, color=AMS_BLUE, edgecolors='white', lw=0.3)

# Regression line
m, b, r, p_val, se = stats.linregress(df['svi_pca'].fillna(0),
                                       df['cooling_access'].fillna(0))
xs = np.linspace(df['svi_pca'].min(), df['svi_pca'].max(), 100)
ax.plot(xs, m*xs+b, color=AMS_RED, lw=2, label=f'OLS: r={r:.3f}, p={p_val:.4f}')

# Highlight top-right quadrant (high SVI, low access)
ax.axvline(df['svi_pca'].median(), color='grey', ls='--', alpha=0.5)
ax.axhline(df['cooling_access'].median(), color='grey', ls='--', alpha=0.5)
ax.text(0.97, 0.03, '⚠ High vulnerability
   Low access',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=10, color=AMS_RED,
        bbox=dict(boxstyle='round', facecolor='#fff3cd', alpha=0.8))

ax.set_xlabel('Social Vulnerability Index (PC1-normalised)', fontsize=11)
ax.set_ylabel('Cooling Access Score (0=poor, 1=excellent)', fontsize=11)
ax.set_title('RQ2: Social Vulnerability vs Cooling Access')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / 'rq2_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/rq2_scatter.png")

pr, pp  = stats.pearsonr(df['svi_pca'].fillna(0), df['cooling_access'].fillna(0))
sr, sp_ = stats.spearmanr(df['svi_pca'].fillna(0), df['cooling_access'].fillna(0))
print(f"\nPearson  r={pr:.4f}, p={pp:.4f}")
print(f"Spearman ρ={sr:.4f}, p={sp_:.4f}")
if pr < 0 and pp < 0.05:
    print("Interpretation: Buurten with higher social vulnerability have significantly "
          "lower cooling access — evidence of double disadvantage.")
elif pp >= 0.05:
    print("Interpretation: No significant linear relationship detected.")

In [ ]:
# ── Quantile regression: SVI ~ cooling_access ─────────────────────────────────
from statsmodels.regression.quantile_regression import QuantReg

qreg_data = df[['svi_pca','cooling_access']].dropna().astype(float)
y_q = qreg_data['svi_pca']
X_q = sm.add_constant(qreg_data['cooling_access'])

taus = [0.10, 0.25, 0.50, 0.75, 0.90]
qr_results = {}

print("Quantile Regression: SVI ~ cooling_access")
print(f"{'Quantile':<12} {'β (cooling_access)':<22} {'95% CI Low':<14} {'95% CI High':<14} {'p-value':<12}")
print('-'*75)
for tau in taus:
    mod = QuantReg(y_q, X_q).fit(q=tau, vcov='iid')
    coef = mod.params['cooling_access']
    ci   = mod.conf_int().loc['cooling_access']
    pv   = mod.pvalues['cooling_access']
    qr_results[tau] = (coef, ci[0], ci[1], pv)
    stars = sig_stars(pv)
    print(f"  τ={tau:<9} {coef:>8.4f}          [{ci[0]:>8.4f}, {ci[1]:>8.4f}]   p={pv:.4f} {stars}")

# ── Plot quantile regression coefficients ────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
coefs = [qr_results[t][0] for t in taus]
ci_lo = [qr_results[t][1] for t in taus]
ci_hi = [qr_results[t][2] for t in taus]

ax.plot(taus, coefs, 'o-', color=AMS_BLUE, lw=2, ms=8, label='Quantile β')
ax.fill_between(taus, ci_lo, ci_hi, alpha=0.2, color=AMS_BLUE, label='95% CI')

# OLS reference
ols_coef = sm.OLS(y_q, X_q).fit().params['cooling_access']
ax.axhline(ols_coef, color=AMS_RED, ls='--', lw=1.5, label=f'OLS β={ols_coef:.4f}')
ax.axhline(0, color='grey', lw=0.8)

ax.set_xlabel('Quantile (τ)')
ax.set_ylabel('β — cooling_access coefficient')
ax.set_title('Quantile Regression: SVI ~ Cooling Access
(across vulnerability distribution)')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'rq2_quantreg.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/rq2_quantreg.png")

In [ ]:
# ── Concentration curve ───────────────────────────────────────────────────────
cc_df = df[['buurtnaam','svi_pca','cooling_access']].dropna().copy()
cc_df = cc_df.sort_values('svi_pca')                 # rank by SVI (ascending)
n_cc  = len(cc_df)
population_share   = np.arange(1, n_cc+1) / n_cc
cumulative_access  = np.cumsum(cc_df['cooling_access']) / cc_df['cooling_access'].sum()
equality_line      = population_share

fig, ax = plt.subplots(figsize=(8, 7))
ax.plot(population_share, cumulative_access, color=AMS_BLUE, lw=2.5,
        label='Concentration Curve (ranked by SVI)')
ax.plot([0,1],[0,1], color='grey', ls='--', lw=1.5, label='Line of Equality')
ax.fill_between(population_share, equality_line, cumulative_access,
                alpha=0.25, color=AMS_BLUE)

# Concentration Index (CI): twice the area between curve and equality line
ci = 2 * np.trapz(cumulative_access - equality_line, population_share)
ax.text(0.05, 0.90, f'Concentration Index = {ci:.4f}',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

ax.set_xlabel('Cumulative share of buurten (ranked by SVI, low→high)')
ax.set_ylabel('Cumulative share of cooling access')
ax.set_title('Concentration Curve — Cooling Access vs Social Vulnerability')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / 'rq2_concentration_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/rq2_concentration_curve.png")

if ci < 0:
    print(f"Concentration Index = {ci:.4f} < 0: cooling access is concentrated among "
          "LESS vulnerable buurten — pro-rich gradient in access.")
elif ci > 0:
    print(f"Concentration Index = {ci:.4f} > 0: cooling access is concentrated among "
          "MORE vulnerable buurten — pro-poor gradient.")
else:
    print(f"Concentration Index ≈ 0: cooling access is equally distributed.")

# ── 4-quadrant gap analysis ───────────────────────────────────────────────────
med_svi  = cc_df['svi_pca'].median()
med_cool = cc_df['cooling_access'].median()

cc_df['quadrant'] = 'Other'
cc_df.loc[(cc_df['svi_pca'] >  med_svi) & (cc_df['cooling_access'] <= med_cool),
          'quadrant'] = '⚠ High SVI, Low Access'
cc_df.loc[(cc_df['svi_pca'] >  med_svi) & (cc_df['cooling_access'] >  med_cool),
          'quadrant'] = '✓ High SVI, Good Access'
cc_df.loc[(cc_df['svi_pca'] <= med_svi) & (cc_df['cooling_access'] <= med_cool),
          'quadrant'] = '△ Low SVI, Low Access'
cc_df.loc[(cc_df['svi_pca'] <= med_svi) & (cc_df['cooling_access'] >  med_cool),
          'quadrant'] = '● Low SVI, Good Access'

df['quadrant'] = df.index.map(cc_df['quadrant'])

print("\n4-Quadrant Distribution:")
print(cc_df['quadrant'].value_counts().to_string())

# Store back for export
gap_count = cc_df['quadrant'].value_counts()

## 12. Composite Heat Vulnerability Index (HVI)

In [ ]:
# ── Normalise heat intensity ──────────────────────────────────────────────────
df['temp_norm'] = pd.to_numeric(df['temp_mean'], errors='coerce')
df['temp_norm'] = df['temp_norm'].fillna(df['temp_norm'].median())
t_min, t_max = df['temp_norm'].min(), df['temp_norm'].max()
df['hi_norm'] = (df['temp_norm'] - t_min) / (t_max - t_min)

# Blend with HI_TOTAAL_S if available
if 'HI_TOTAAL_S' in df.columns:
    hi_s = pd.to_numeric(df['HI_TOTAAL_S'], errors='coerce')
    if hi_s.notna().sum() > 50:
        hi_s_norm = (hi_s - hi_s.min()) / (hi_s.max() - hi_s.min())
        df['hi_norm'] = df['hi_norm'] * 0.5 + hi_s_norm.fillna(df['hi_norm']) * 0.5
        print("hi_norm = blend of temp_mean + HI_TOTAAL_S (50/50)")
    else:
        print("hi_norm = temp_mean only (insufficient HI_TOTAAL_S coverage)")
else:
    print("hi_norm = temp_mean only")

# ── HVI = 0.40×hi_norm + 0.40×svi_pca + 0.20×(1-cooling_access) ─────────────
W_HI   = 0.40
W_SVI  = 0.40
W_COOL = 0.20

df['hvi'] = (W_HI   * df['hi_norm']
           + W_SVI  * df['svi_pca']
           + W_COOL * (1 - df['cooling_access']))

# ── Quintile tiers ────────────────────────────────────────────────────────────
df['hvi_tier'] = pd.qcut(df['hvi'], q=5, labels=[1,2,3,4,5]).astype(float)

print(f"\nHVI summary:")
print(df[['hi_norm','svi_pca','cooling_access','hvi','hvi_tier']].describe().round(3))

tier_counts = df['hvi_tier'].value_counts().sort_index()
print("\nHVI quintile tier sizes:")
print(tier_counts.to_string())

In [ ]:
# ── HVI overview visualisation ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. HVI distribution
axes[0,0].hist(df['hvi'], bins=30, color=AMS_BLUE, alpha=0.8, edgecolor='white')
axes[0,0].set_xlabel('HVI'); axes[0,0].set_ylabel('Count')
axes[0,0].set_title('HVI Distribution')

# 2. Component correlations
comp_df = df[['hi_norm','svi_pca','cooling_access','hvi']].dropna()
sns.heatmap(comp_df.corr(), annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, ax=axes[0,1], linewidths=0.5)
axes[0,1].set_title('Component Correlations')
axes[0,1].tick_params(axis='x', rotation=30, labelsize=8)

# 3. Tier box plots for each component
tier_df = df[['hvi_tier','hi_norm','svi_pca','cooling_access']].dropna()
tier_df.boxplot(column='hi_norm', by='hvi_tier', ax=axes[0,2],
    boxprops=dict(color=AMS_BLUE), medianprops=dict(color=AMS_RED))
axes[0,2].set_title('hi_norm by HVI Tier')
axes[0,2].set_xlabel('Tier'); axes[0,2].set_ylabel('hi_norm')
plt.sca(axes[0,2]); plt.title('hi_norm by HVI Tier')

tier_df.boxplot(column='svi_pca', by='hvi_tier', ax=axes[1,0],
    boxprops=dict(color=AMS_BLUE), medianprops=dict(color=AMS_RED))
axes[1,0].set_title('svi_pca by HVI Tier')
axes[1,0].set_xlabel('Tier'); plt.sca(axes[1,0]); plt.title('svi_pca by HVI Tier')

tier_df.boxplot(column='cooling_access', by='hvi_tier', ax=axes[1,1],
    boxprops=dict(color=AMS_BLUE), medianprops=dict(color=AMS_RED))
axes[1,1].set_title('Cooling Access by HVI Tier')
axes[1,1].set_xlabel('Tier'); plt.sca(axes[1,1]); plt.title('Cooling by HVI Tier')

# 6. Top-20 buurten bar chart
top20 = df.nlargest(20, 'hvi')[['buurtnaam','hvi']].sort_values('hvi')
axes[1,2].barh(top20['buurtnaam'], top20['hvi'], color=AMS_BLUE)
axes[1,2].set_xlabel('HVI')
axes[1,2].set_title('Top-20 Most Vulnerable Buurten')
axes[1,2].tick_params(labelsize=7)

plt.suptitle('Composite HVI Overview', fontsize=14)
plt.tight_layout()
plt.savefig(OUT_DIR / 'hvi_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/hvi_overview.png")

In [ ]:
# ── Validate against HI_TOTAAL_S ─────────────────────────────────────────────
if 'HI_TOTAAL_S' in df.columns:
    val_df = df[['hvi', 'HI_TOTAAL_S']].dropna()
    val_df['HI_TOTAAL_S'] = pd.to_numeric(val_df['HI_TOTAAL_S'], errors='coerce')
    val_df = val_df.dropna()

    r_val, p_val_r = stats.pearsonr(val_df['hvi'], val_df['HI_TOTAAL_S'])
    print(f"HVI vs HI_TOTAAL_S: Pearson r={r_val:.4f}, p={p_val_r:.6f}, n={len(val_df)}")
    if r_val > 0.5 and p_val_r < 0.05:
        print("→ Strong positive convergent validity with Klimaatrisicokaarten scores.")
    else:
        print("→ Moderate/weak correlation — review component weights.")

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(val_df['HI_TOTAAL_S'], val_df['hvi'],
               s=20, alpha=0.6, color=AMS_BLUE, edgecolors='white')
    m_v, b_v, _, _, _ = stats.linregress(val_df['HI_TOTAAL_S'], val_df['hvi'])
    xs_v = np.linspace(val_df['HI_TOTAAL_S'].min(), val_df['HI_TOTAAL_S'].max(), 100)
    ax.plot(xs_v, m_v*xs_v+b_v, color=AMS_RED, lw=2,
            label=f'r={r_val:.3f}')
    ax.set_xlabel('HI_TOTAAL_S (Klimaatrisicokaarten)')
    ax.set_ylabel('Composite HVI')
    ax.set_title('Convergent Validity: HVI vs HI_TOTAAL_S')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'hvi_validation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: outputs/hvi_validation.png")
else:
    print("HI_TOTAAL_S not available — skipping convergent validity check.")

# ── Priority ranking table ────────────────────────────────────────────────────
rank_cols = ['buurtnaam', 'hvi', 'hvi_tier', 'hi_norm', 'svi_pca', 'cooling_access']
rank_cols = [c for c in rank_cols if c in df.columns]
priority_df = df.nlargest(20, 'hvi')[rank_cols].reset_index(drop=True)
priority_df.index = priority_df.index + 1
priority_df.index.name = 'Rank'
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(priority_df)

## 13. Sensitivity & Bootstrap Analysis

In [ ]:
# ── Weight perturbation analysis ─────────────────────────────────────────────
from itertools import product as iproduct

base_w = {'hi': 0.40, 'svi': 0.40, 'cool': 0.20}
deltas = np.arange(-0.10, 0.11, 0.05)   # ±0.10 in 0.05 steps

def compute_hvi(hi, svi, cool, w_hi, w_svi, w_cool):
    total = w_hi + w_svi + w_cool
    return (w_hi/total)*hi + (w_svi/total)*svi + (w_cool/total)*(1-cool)

perturb_results = []
base_hvi = compute_hvi(df['hi_norm'], df['svi_pca'], df['cooling_access'],
                        base_w['hi'], base_w['svi'], base_w['cool'])
base_rank = base_hvi.rank(ascending=False)

for d_hi in deltas:
    for d_svi in deltas:
        w_hi  = base_w['hi']  + d_hi
        w_svi = base_w['svi'] + d_svi
        w_cool = 1.0 - w_hi - w_svi
        if w_hi < 0.05 or w_svi < 0.05 or w_cool < 0.05:
            continue
        if w_hi + w_svi + w_cool <= 0:
            continue
        pert_hvi  = compute_hvi(df['hi_norm'], df['svi_pca'], df['cooling_access'],
                                 w_hi, w_svi, w_cool)
        pert_rank = pert_hvi.rank(ascending=False)
        rho, _ = stats.spearmanr(base_rank, pert_rank)
        perturb_results.append({
            'w_hi': round(w_hi, 3), 'w_svi': round(w_svi, 3),
            'w_cool': round(w_cool, 3), 'spearman_rho': rho
        })

pert_df = pd.DataFrame(perturb_results)
print(f"Weight perturbation: {len(pert_df)} weight sets tested")
print(f"Spearman ρ with base ranking: min={pert_df['spearman_rho'].min():.4f}, "
      f"mean={pert_df['spearman_rho'].mean():.4f}, max={pert_df['spearman_rho'].max():.4f}")
print(f"Sets with ρ < 0.90: {(pert_df['spearman_rho'] < 0.90).sum()}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(pert_df['spearman_rho'], bins=25, color=AMS_BLUE, alpha=0.8, edgecolor='white')
ax.axvline(0.90, color=AMS_RED, ls='--', lw=2, label='ρ=0.90 threshold')
ax.axvline(pert_df['spearman_rho'].mean(), color=AMS_ORANGE, ls='--',
           lw=2, label=f"Mean ρ={pert_df['spearman_rho'].mean():.3f}")
ax.set_xlabel('Spearman ρ vs base ranking')
ax.set_ylabel('Count')
ax.set_title('HVI Ranking Stability Under Weight Perturbation (±0.10)')
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'sensitivity_weights.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/sensitivity_weights.png")

In [ ]:
# ── Bootstrap: OLS regression coefficients ───────────────────────────────────
np.random.seed(42)
N_BOOT = 1000

boot_phys = [v for v in PHYS_VARS[1:] + CTRL_VARS if v in df.columns]
X_boot_full = df[boot_phys + soc_reg + ['temp_mean']].copy()
for c in X_boot_full.columns:
    X_boot_full[c] = pd.to_numeric(X_boot_full[c], errors='coerce').fillna(
        pd.to_numeric(X_boot_full[c], errors='coerce').median())

all_pred_cols = boot_phys + soc_reg
boot_coefs = {col: [] for col in all_pred_cols}

print(f"Running bootstrap ({N_BOOT} replications) …", end='')
n_obs = len(X_boot_full)
for _ in range(N_BOOT):
    idx = np.random.choice(n_obs, n_obs, replace=True)
    X_b = sm.add_constant(X_boot_full.iloc[idx][all_pred_cols].astype(float))
    y_b = X_boot_full.iloc[idx]['temp_mean']
    try:
        res_b = sm.OLS(y_b, X_b).fit(disp=0)
        for col in all_pred_cols:
            if col in res_b.params:
                boot_coefs[col].append(res_b.params[col])
    except Exception:
        pass
print(' done')

ci_rows = []
for col in all_pred_cols:
    arr = np.array(boot_coefs[col])
    if len(arr) < 10:
        continue
    lo, hi_b = np.percentile(arr, [2.5, 97.5])
    ci_rows.append({
        'Variable': col,
        'Boot_mean': arr.mean(),
        'CI_lo_2.5': lo,
        'CI_hi_97.5': hi_b,
        'Significant': 'Yes' if (lo > 0 or hi_b < 0) else 'No'
    })

boot_ci_df = pd.DataFrame(ci_rows)
print("\nBootstrap 95% CI for OLS coefficients (Model B):")
with pd.option_context('display.float_format', '{:.4f}'.format):
    display(boot_ci_df)

# ── Forest plot ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, max(5, len(boot_ci_df)*0.45)))
y_pos = np.arange(len(boot_ci_df))

for i, row in boot_ci_df.iterrows():
    color = AMS_BLUE if row['Significant'] == 'Yes' else 'grey'
    ax.plot([row['CI_lo_2.5'], row['CI_hi_97.5']], [i, i], lw=2.5, color=color)
    ax.scatter(row['Boot_mean'], i, s=60, zorder=5, color=color)

ax.axvline(0, color='black', lw=1, ls='--')
short_names = [r.replace('percentage','pct_').replace('Gemiddelde','')[:25]
               for r in boot_ci_df['Variable']]
ax.set_yticks(range(len(short_names)))
ax.set_yticklabels(short_names, fontsize=9)
ax.set_xlabel('OLS Coefficient')
ax.set_title(f'Bootstrap 95% CI — OLS Coefficients (n={N_BOOT} replications)')
blue_p  = mpatches.Patch(color=AMS_BLUE, label='Significant (CI excludes 0)')
grey_p  = mpatches.Patch(color='grey',   label='Not significant')
ax.legend(handles=[blue_p, grey_p], fontsize=9)
plt.tight_layout()
plt.savefig(OUT_DIR / 'bootstrap_ci.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: outputs/bootstrap_ci.png")

In [ ]:
# ── Bootstrap: HVI tier boundary stability ────────────────────────────────────
hvi_vals = df['hvi'].dropna().values
boundaries_boot = {q: [] for q in [0.20, 0.40, 0.60, 0.80]}

np.random.seed(99)
for _ in range(N_BOOT):
    sample = np.random.choice(hvi_vals, len(hvi_vals), replace=True)
    for q in boundaries_boot:
        boundaries_boot[q].append(np.quantile(sample, q))

print("Bootstrap 95% CI for HVI tier boundaries:")
print(f"{'Tier boundary (quantile)':<28} {'Mean':>8} {'CI 2.5%':>10} {'CI 97.5%':>10}")
print('-'*58)
for q, label in [(0.20,'Tier 1/2'), (0.40,'Tier 2/3'), (0.60,'Tier 3/4'), (0.80,'Tier 4/5')]:
    arr = np.array(boundaries_boot[q])
    lo_b, hi_b = np.percentile(arr, [2.5, 97.5])
    print(f"  {label} (Q{q:.0%})            {arr.mean():>8.4f} {lo_b:>10.4f} {hi_b:>10.4f}")

print("\nInterpretation: Narrow CIs indicate stable tier boundaries across resamples.")

## 14. Export

In [ ]:
# ── Compile output dataframe ──────────────────────────────────────────────────
export_cols = ['buurtcode', 'buurtnaam', 'hvi', 'hvi_tier',
               'hi_norm', 'svi_pca', 'cooling_access', 'quadrant']

# Add HI columns if present
for col in HI_COLS + ['lisa_cluster']:
    if col in df.columns:
        export_cols.append(col)

export_cols = [c for c in export_cols if c in df.columns]
df_export = df[export_cols].copy()
df_export['hvi']          = df_export['hvi'].round(6)
df_export['hi_norm']      = df_export['hi_norm'].round(6)
df_export['svi_pca']      = df_export['svi_pca'].round(6)
df_export['cooling_access'] = df_export['cooling_access'].round(6)

# ── Write hvi_scores.csv ──────────────────────────────────────────────────────
csv_out = OUT_DIR / 'hvi_scores.csv'
# Also overwrite the charlie/ copy referenced by the notebook spec
charlie_hvi_out = DATA_DIR / 'hvi_scores.csv'

df_export.to_csv(csv_out, index=False)
df_export.to_csv(charlie_hvi_out, index=False)
print(f"Written: {csv_out}  ({len(df_export)} rows, {len(df_export.columns)} cols)")
print(f"Written: {charlie_hvi_out}")

# ── Write hvi_map.geojson (if geopandas available) ────────────────────────────
GEOJSON_OUT         = OUT_DIR  / 'hvi_map.geojson'
GEOJSON_CHARLIE_OUT = DATA_DIR / 'hvi_map.geojson'

if HAS_GEO and gdf is not None:
    gdf_out = gdf.copy()
    # Merge computed scores
    merge_on = 'buurtcode' if 'buurtcode' in df_export.columns else 'buurtnaam'
    gdf_out = gdf_out.merge(
        df_export.drop(columns=[c for c in df_export.columns
                                 if c in gdf_out.columns and c != merge_on]),
        on=merge_on, how='left'
    )
    # Re-project to WGS84 for GeoJSON
    if gdf_out.crs is not None and gdf_out.crs.to_epsg() != 4326:
        gdf_out = gdf_out.to_crs(epsg=4326)

    gdf_out.to_file(str(GEOJSON_OUT),         driver='GeoJSON')
    gdf_out.to_file(str(GEOJSON_CHARLIE_OUT), driver='GeoJSON')
    print(f"Written: {GEOJSON_OUT}")
    print(f"Written: {GEOJSON_CHARLIE_OUT}")
else:
    print("Skipping GeoJSON export (geopandas not available or no geometry loaded).")
    print("  → To enable: pip install geopandas fiona shapely")
    print(f"  → CSV export is complete: {csv_out}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("EXPORT SUMMARY")
print("="*60)
print(f"  hvi_scores.csv   : {len(df_export)} buurten × {len(df_export.columns)} variables")
print(f"  Columns          : {list(df_export.columns)}")
if HAS_GEO and gdf is not None:
    print(f"  hvi_map.geojson  : {len(gdf_out)} features (WGS84)")
print("\nPipeline complete.")